## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [2]:
import json
from pathlib import Path


def find_data_file(*parts):
    """Find lesson data from common notebook working directories."""
    filename = Path(*parts).name
    candidates = [
        Path("../../data").joinpath(*parts),
        Path("../../../data").joinpath(*parts),
        Path("data").joinpath(*parts),
        Path("Assignments_Completed/03-Data_Manager/data").joinpath(*parts),
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = [path for path in Path.cwd().rglob(filename) if path.parts[-len(parts):] == parts]
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find {'/'.join(parts)} from {Path.cwd()}")


data_path = find_data_file("ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [3]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [4]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [5]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [6]:
lod_path = find_data_file("lod", "railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [7]:
# Write collection_bbox(features) and apply to all four LOD files

def collection_bbox(features):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a list of GeoJSON LineString features.
    """
    lon_min = lat_min = float("inf")
    lon_max = lat_max = float("-inf")

    for feature in features:
        feature_lon_min, feature_lat_min, feature_lon_max, feature_lat_max = feature_bbox(feature)
        lon_min = min(lon_min, feature_lon_min)
        lat_min = min(lat_min, feature_lat_min)
        lon_max = max(lon_max, feature_lon_max)
        lat_max = max(lat_max, feature_lat_max)

    return [lon_min, lat_min, lon_max, lat_max]


lod_files = {
    "coarse": "railroads_coarse.geojson",
    "medium": "railroads_medium.geojson",
    "fine": "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

lod_collections = {}
collection_bboxes = {}

for level, filename in lod_files.items():
    path = find_data_file("lod", filename)
    with open(path) as f:
        lod_collections[level] = json.load(f)

    features = lod_collections[level]["features"]
    bbox = collection_bbox(features)
    collection_bboxes[level] = bbox
    print(f"{level:10} {len(features):6,} features  bbox: {bbox}")

print()
print("Do they cover the same extent?")
print("No. The coarse LOD covers a smaller extent because it filters to only high-priority features.")
print("The medium and fine LODs match, and extra_fine is almost identical with a tiny latitude difference from simplification.")

coarse      2,845 features  bbox: [-123.014722, -41.475186, 150.961667, 60.976516]
medium     25,413 features  bbox: [-150.112222, -51.894722, 179.357778, 69.604375]
fine       25,413 features  bbox: [-150.112222, -51.894722, 179.357778, 69.604375]
extra_fine 25,413 features  bbox: [-150.112222, -51.895278, 179.357778, 69.604375]

Do they cover the same extent?
No. The coarse LOD covers a smaller extent because it filters to only high-priority features.
The medium and fine LODs match, and extra_fine is almost identical with a tiny latitude difference from simplification.


## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [8]:
# Find the 5 features with the largest bounding box area in railroads_fine.geojson

def bbox_area(bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    return (lon_max - lon_min) * (lat_max - lat_min)


fine_features = lod_collections.get("fine")
if fine_features is None:
    with open(find_data_file("lod", "railroads_fine.geojson")) as f:
        fine_features = json.load(f)

largest = []
for feature in fine_features["features"]:
    bbox = feature_bbox(feature)
    largest.append((bbox_area(bbox), bbox, feature))

largest = sorted(largest, reverse=True, key=lambda item: item[0])[:5]

for rank, (area, bbox, feature) in enumerate(largest, start=1):
    props = feature["properties"]
    print(
        f"{rank}. area={area:.2f}, "
        f"category={props.get('category')}, "
        f"continent={props.get('continent')}, "
        f"bbox={bbox}"
    )

print()
print("These make sense: the largest boxes belong to features that span large longitude and latitude ranges.")
print("A large bbox does not necessarily mean the railroad is physically longest; it means its axis-aligned extent is large.")

1. area=29.31, category=0, continent=Asia, bbox=[90.609351, 29.645107, 94.942815, 36.409271]
2. area=18.65, category=0, continent=Oceania, bbox=[132.255704, -23.549819, 134.323448, -14.530934]
3. area=17.70, category=3, continent=South America, bbox=[-64.094167, -26.192499, -58.156944, -23.210555]
4. area=17.39, category=2, continent=Europe, bbox=[14.021914, 54.802728, 20.000001, 57.711109]
5. area=12.35, category=2, continent=South America, bbox=[-49.137778, -5.491666, -44.372222, -2.899167]

These make sense: the largest boxes belong to features that span large longitude and latitude ranges.
A large bbox does not necessarily mean the railroad is physically longest; it means its axis-aligned extent is large.


## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

**Answer:** Two railroad features can share the same bounding box if they touch the same western, eastern, southern, and northern extremes but travel through that rectangle differently. For example, one railroad might run diagonally from the southwest corner to the northeast corner, while another might go north along the western edge, east across the top, then down along the eastern edge.

This does not break viewport culling. Bounding-box culling is intentionally conservative: if a bbox intersects the viewport, we keep the feature for the next rendering/filtering step. Identical boxes can create false positives, but they will not incorrectly remove a visible feature.

## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.